# Week 9: LOZO Medium-Scale Replication (Prime Intellect, 80GB)

This notebook reproduces the LOZO medium-model pipeline in a week8-style, script-centric workflow:

1. **Environment + repo setup** (Prime-friendly paths and CUDA checks)
2. **Data preparation** (k-shot splits for K=16 and K=512)
3. **Smoke suite** (quick LOZO vs MeZO sanity checks)
4. **Full suite** (multi-task, multi-seed runs)
5. **Aggregation + plots** (LOZO vs MeZO comparison tables/figures)

Primary reference code: [optsuite/LOZO](https://github.com/optsuite/LOZO) (`medium_models`).

---
## §0 — Environment and Runtime Profile

This section defines reproducible paths and checks GPU readiness for a single 80GB Prime Intellect machine.

**Assumptions**
- Run from the `stat4830` repo root.
- Environment is prepared with `uv sync`.
- One CUDA GPU is available (80GB target profile).

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
LOZO_ROOT = REPO_ROOT / "external" / "LOZO"
MEDIUM_DIR = LOZO_ROOT / "medium_models"

RUN_TAG = "week9_lozo_medium_prime80gb"
SMOKE_RUN_ROOT = REPO_ROOT / "results" / RUN_TAG / "smoke"
FULL_RUN_ROOT = REPO_ROOT / "results" / RUN_TAG / "full"

TASKS_FULL = ["SST-2", "sst-5", "SNLI", "MNLI", "RTE", "trec"]
SEEDS_FULL = [13, 21, 42, 87, 100]
K_VALUES_FULL = [16, 512]

print("REPO_ROOT:", REPO_ROOT)
print("LOZO_ROOT:", LOZO_ROOT)
print("RUN_TAG:", RUN_TAG)

In [ ]:
import torch

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    prop = torch.cuda.get_device_properties(idx)
    print("GPU:", torch.cuda.get_device_name(idx))
    print("VRAM (GB):", round(prop.total_memory / (1024 ** 3), 2))
else:
    print("No CUDA device detected. Full medium runs will be very slow on CPU.")

In [ ]:
# Clone LOZO if needed, then print the pinned commit for reproducibility.
if not LOZO_ROOT.exists():
    LOZO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "https://github.com/optsuite/LOZO", str(LOZO_ROOT)
    ], check=True)

commit = subprocess.check_output(
    ["git", "-C", str(LOZO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
print("LOZO commit:", commit)
print("medium_models exists:", MEDIUM_DIR.exists())

---
## §1 — Verify LOZO Medium Pipeline

LOZO upstream entry points used in this notebook:
- `medium_models/lozo.sh`
- `medium_models/mezo.sh`
- `medium_models/tools/generate_k_shot_data.py`
- `medium_models/tools/gather_result.py`

This notebook does not duplicate training internals. It orchestrates upstream scripts and parses outputs.

In [ ]:
required = [
    MEDIUM_DIR / "lozo.sh",
    MEDIUM_DIR / "mezo.sh",
    MEDIUM_DIR / "tools" / "generate_k_shot_data.py",
    MEDIUM_DIR / "tools" / "gather_result.py",
]

for p in required:
    print(f"{p}:", "OK" if p.exists() else "MISSING")

---
## §2 — Data Prep (K-Shot Splits)

LOZO medium-model experiments use LM-BFF style k-shot data. Upstream defaults in the paper:
- `K in {16, 512}`
- Seeds: `{13, 21, 42, 87, 100}`
- Dataset root under `medium_models/data/k-shot-1k-test`

Run the next cell once per machine.

In [ ]:
# Download datasets if needed.
subprocess.run(["bash", "download_dataset.sh"], cwd=MEDIUM_DIR / "data", check=False)

# Generate splits for K=16 and K=512 using upstream tool defaults.
for k in [16, 512]:
    cmd = [
        sys.executable,
        "tools/generate_k_shot_data.py",
        "--mode", "k-shot-1k-test",
        "--k", str(k),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=MEDIUM_DIR, check=True)

print("Data prep complete.")

In [ ]:
probe_paths = [
    MEDIUM_DIR / "data" / "k-shot-1k-test" / "SST-2" / "16-42",
    MEDIUM_DIR / "data" / "k-shot-1k-test" / "RTE" / "512-13",
]
for p in probe_paths:
    print(p, "->", "OK" if p.exists() else "MISSING")

---
## §3 — Smoke Suite (LOZO vs MeZO)

We run a short, resume-safe smoke suite using `src.scripts.run_lozo_medium_suite`.

Smoke profile defaults:
- Tasks: `SST-2`, `RTE`
- `K=16`
- Seed: `42`
- Methods: `mezo`, `lozo`
- Short schedule: `--smoke-steps 1000 --smoke-eval-steps 100`

This validates environment, paths, logs, and completion markers before full runs.

In [ ]:
SMOKE_RUN_ROOT.mkdir(parents=True, exist_ok=True)

cmd_smoke_dry = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--run-root", str(SMOKE_RUN_ROOT),
    "--profile", "smoke",
    "--dry-run",
]
print(" ".join(cmd_smoke_dry))
subprocess.run(cmd_smoke_dry, check=True, cwd=REPO_ROOT)

manifest_path = SMOKE_RUN_ROOT / "manifest.json"
print("Manifest exists:", manifest_path.exists())
if manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    print("Planned smoke runs:", m.get("num_runs"))

In [ ]:
# Toggle this to launch smoke runs.
RUN_SMOKE = False

cmd_smoke = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--run-root", str(SMOKE_RUN_ROOT),
    "--profile", "smoke",
]

if RUN_SMOKE:
    print(" ".join(cmd_smoke))
    subprocess.run(cmd_smoke, check=False, cwd=REPO_ROOT)
else:
    print("RUN_SMOKE is False. Set RUN_SMOKE=True to start smoke training.")
    print("Command preview:")
    print(" ".join(cmd_smoke))

In [ ]:
smoke_summary_path = SMOKE_RUN_ROOT / "summary.json"
if smoke_summary_path.exists():
    smoke = json.loads(smoke_summary_path.read_text())
    print("Smoke counts:", smoke.get("counts", {}))
    for rec in smoke.get("records", [])[:6]:
        print(
            rec.get("method"), rec.get("task"),
            f"k={rec.get('k')} seed={rec.get('seed')}",
            "status=", rec.get("status"),
            "metrics=", rec.get("metrics", {}),
        )
else:
    print("No smoke summary yet. Run the execution cell first.")

---
## §4 — Full Medium-Scale Suite

Full profile defaults map to the paper-style medium setup:
- Tasks: `SST-2`, `sst-5`, `SNLI`, `MNLI`, `RTE`, `trec`
- K values: `16`, `512`
- Seeds: `13`, `21`, `42`, `87`, `100`
- Methods: `mezo`, `lozo`
- Upstream schedule defaults from scripts: `STEP=100000`, `EVAL_STEP=10000`

Run matrix size (default): `6 tasks x 2 K x 5 seeds x 2 methods = 120 runs`.

In [ ]:
FULL_RUN_ROOT.mkdir(parents=True, exist_ok=True)

cmd_full_dry = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--run-root", str(FULL_RUN_ROOT),
    "--profile", "full",
    "--dry-run",
]
print(" ".join(cmd_full_dry))
subprocess.run(cmd_full_dry, check=True, cwd=REPO_ROOT)

full_manifest = FULL_RUN_ROOT / "manifest.json"
if full_manifest.exists():
    m = json.loads(full_manifest.read_text())
    print("Planned full runs:", m.get("num_runs"))
    print("Methods:", sorted({r['method'] for r in m['runs']}))
    print("Tasks:", sorted({r['task'] for r in m['runs']}))

In [ ]:
# Toggle this to launch full runs.
RUN_FULL = False

cmd_full = [
    sys.executable,
    "-m",
    "src.scripts.run_lozo_medium_suite",
    "--lozo-root", str(LOZO_ROOT),
    "--run-root", str(FULL_RUN_ROOT),
    "--profile", "full",
]

if RUN_FULL:
    print(" ".join(cmd_full))
    subprocess.run(cmd_full, check=False, cwd=REPO_ROOT)
else:
    print("RUN_FULL is False. Set RUN_FULL=True to start full training.")
    print("Command preview:")
    print(" ".join(cmd_full))

---
## §5 — Aggregate Results and Compare LOZO vs MeZO

Aggregation reads `summary.json` produced by the orchestration script and computes per-task comparisons.

By default, this section reads from `FULL_RUN_ROOT/summary.json`. You can point it to smoke outputs for quick testing.

In [ ]:
SUMMARY_SOURCE = FULL_RUN_ROOT / "summary.json"
# For quick debugging, you can switch to: SUMMARY_SOURCE = SMOKE_RUN_ROOT / "summary.json"


def choose_metric(metrics: dict[str, float]) -> tuple[str | None, float | None]:
    if not metrics:
        return None, None
    ordered = sorted(metrics.items())
    for key, value in ordered:
        if "acc" in key.lower():
            return key, value
    return ordered[0]


def load_records(summary_path: Path) -> list[dict]:
    if not summary_path.exists():
        return []
    payload = json.loads(summary_path.read_text())
    rows = []
    for rec in payload.get("records", []):
        if rec.get("status") not in {"completed", "skipped"}:
            continue
        metric_name, metric_val = choose_metric(rec.get("metrics", {}))
        if metric_val is None:
            continue
        rows.append(
            {
                "method": rec.get("method"),
                "task": rec.get("task"),
                "k": int(rec.get("k")),
                "seed": int(rec.get("seed")),
                "metric_name": metric_name,
                "metric": float(metric_val),
                "status": rec.get("status"),
                "result_dir": rec.get("result_dir"),
            }
        )
    return rows


def aggregate(rows: list[dict]) -> list[dict]:
    buckets: dict[tuple[str, int, str], list[float]] = defaultdict(list)
    for r in rows:
        buckets[(r["task"], r["k"], r["method"])].append(r["metric"])

    out = []
    for (task, k, method), vals in sorted(buckets.items()):
        mean = sum(vals) / len(vals)
        var = sum((v - mean) ** 2 for v in vals) / len(vals)
        out.append(
            {
                "task": task,
                "k": k,
                "method": method,
                "n": len(vals),
                "mean": mean,
                "std": var ** 0.5,
            }
        )
    return out


records = load_records(SUMMARY_SOURCE)
agg = aggregate(records)
print(f"summary path: {SUMMARY_SOURCE}")
print(f"records with metrics: {len(records)}")
print(f"aggregated groups: {len(agg)}")

In [ ]:
if not agg:
    print("No aggregated results yet. Run smoke/full suite first.")
else:
    print(f"{'Task':<10} {'K':>4} {'Method':<8} {'n':>3} {'mean':>10} {'std':>10}")
    print("-" * 55)
    for r in agg:
        print(f"{r['task']:<10} {r['k']:>4} {r['method']:<8} {r['n']:>3} {r['mean']:>10.4f} {r['std']:>10.4f}")

    print("\nLOZO - MeZO deltas by task/K")
    print(f"{'Task':<10} {'K':>4} {'LOZO':>10} {'MeZO':>10} {'Delta':>10}")
    print("-" * 50)
    grouped = defaultdict(dict)
    for r in agg:
        grouped[(r['task'], r['k'])][r['method']] = r['mean']

    for (task, k), d in sorted(grouped.items()):
        lozo = d.get("lozo")
        mezo = d.get("mezo")
        if lozo is None or mezo is None:
            continue
        print(f"{task:<10} {k:>4} {lozo:>10.4f} {mezo:>10.4f} {lozo - mezo:>10.4f}")

In [ ]:
if not agg:
    print("No data to plot yet.")
else:
    by_k = defaultdict(list)
    for r in agg:
        by_k[r["k"]].append(r)

    fig, axes = plt.subplots(1, max(1, len(by_k)), figsize=(7 * max(1, len(by_k)), 5), squeeze=False)

    for ax, k in zip(axes[0], sorted(by_k.keys())):
        rows_k = by_k[k]
        tasks = sorted({r["task"] for r in rows_k})
        x = list(range(len(tasks)))
        mezo = []
        lozo = []
        for t in tasks:
            m = next((r for r in rows_k if r["task"] == t and r["method"] == "mezo"), None)
            l = next((r for r in rows_k if r["task"] == t and r["method"] == "lozo"), None)
            mezo.append(m["mean"] if m else float("nan"))
            lozo.append(l["mean"] if l else float("nan"))

        width = 0.4
        ax.bar([i - width / 2 for i in x], mezo, width=width, label="MeZO")
        ax.bar([i + width / 2 for i in x], lozo, width=width, label="LOZO")
        ax.set_title(f"Task means (K={k})")
        ax.set_xticks(x)
        ax.set_xticklabels(tasks, rotation=35, ha="right")
        ax.set_ylabel("Metric")
        ax.legend()

    fig.tight_layout()
    out_fig = FULL_RUN_ROOT / "lozo_vs_mezo_task_means.png"
    out_fig.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_fig, dpi=180)
    plt.show()
    print("Saved:", out_fig)

---
## §6 — Reproducibility, Runtime, and Recovery

### Resume behavior
- The runner checks completion markers (`test_results_<TASK>.txt`) and skips finished runs.
- Re-run the same command with the same `--run-root` to resume.
- Use `--force` to re-run completed configurations.

### Runtime guidance (single 80GB GPU)
- **Smoke profile**: typically minutes to a couple of hours depending on download/cache state.
- **Full profile (120 runs)**: can take multiple days. Run in stages and monitor `summary.json`.
- Keep a persistent cache for model/data weights between machine restarts.

### Practical recovery commands
```bash
# Resume smoke
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --run-root results/week9_lozo_medium_prime80gb/smoke \
  --profile smoke

# Resume full suite
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --run-root results/week9_lozo_medium_prime80gb/full \
  --profile full

# Re-run everything in full profile
uv run python -m src.scripts.run_lozo_medium_suite \
  --lozo-root external/LOZO \
  --run-root results/week9_lozo_medium_prime80gb/full \
  --profile full \
  --force
```

### Common issues
- `data/k-shot-1k-test/...` missing: run the data-prep section again.
- Missing CUDA or OOM: reduce batch size (`--mezo-bs`, `--lozo-bs`) or run a smaller subset (`--tasks`, `--seeds`, `--k-values`).
- Interrupted jobs: rerun same command; completed runs are skipped automatically.